In [ ]:
import os
import glob
import hashlib
import numpy as np
import pandas as pd
from tqdm import tqdm
from bisect import bisect_left
from collections import Counter
from sklearn.preprocessing import minmax_scale
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors


# ==========================================================
# DATA IMPORT
# ==========================================================

def importation_csv():
    csv_files = glob.glob('./TrafficLabelling/*.csv')
    combined_df = pd.DataFrame()
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, encoding='cp1252')
        combined_df = pd.concat([combined_df, df])
    combined_df.replace([np.inf, -np.inf], np.nan, inplace=True)
    return combined_df


def creation_X_Y_ip(data):
    data_without_nan = data.values[~pd.isna(data.values).any(axis=1)]

    source_port = np.asarray(data_without_nan[1:, 2]).reshape(-1, 1)
    dest_port   = np.asarray(data_without_nan[1:, 4]).reshape(-1, 1)

    source_ip = np.asarray(data_without_nan[1:, 1]).reshape(-1, 1)
    dest_ip   = np.asarray(data_without_nan[1:, 3]).reshape(-1, 1)

    X = data_without_nan[1:, 7:]  # features + label (last col)
    ohe = pd.get_dummies(np.asarray(data_without_nan[1:, 5]).astype(int), dtype=int)

    Y = np.asarray(X[:, -1]).reshape(-1, 1)
    X = X[:, :-1]

    # prepend ports
    X = np.concatenate((source_port, X), axis=1)
    X = np.concatenate((dest_port, X), axis=1)

    # scale numeric
    X = minmax_scale(X, axis=0)

    # prepend protocol OHE
    X = np.concatenate((ohe.values, X), axis=1)

    return X.astype(np.float32), Y, source_ip, dest_ip, source_port, dest_port


def choix_donnees_entrainement_70_30(X, Y, sip, dip, sport, dport):
    le_y = preprocessing.LabelEncoder()
    Y = le_y.fit_transform(Y.ravel())

    # ip encoders (keep 1 encoder each like your original)
    le_ip = preprocessing.LabelEncoder()
    sip = le_ip.fit_transform(sip.ravel()).astype(int)
    dip = le_ip.fit_transform(dip.ravel()).astype(int)

    sport = np.asarray(sport).ravel().astype(int)
    dport = np.asarray(dport).ravel().astype(int)

    return train_test_split(
        X, Y, sip, dip, sport, dport,
        random_state=843, test_size=0.3, stratify=Y
    )


# ==========================================================
# HASH / BINARY SEARCH (original spirit)
# ==========================================================

def hash_flow(sip, dip, dport):
    # you said: couple sip/dip/dp
    min_ip = min(sip, dip)
    max_ip = max(sip, dip)
    # include dport only (as in paper)
    return hashlib.sha256(f"{min_ip}{max_ip}{dport}".encode()).hexdigest()


def binary_search_hash(hashed_data_with_indices, target):
    keys = [h[0] for h in hashed_data_with_indices]
    idx = bisect_left(keys, target)
    if idx != len(keys) and keys[idx] == target:
        return hashed_data_with_indices[idx][1]
    return -1


# ==========================================================
# STREAMING "SMOTE-like" FOR CLASSES 8 & 9
#   - DOES NOT REORDER TIME
#   - INJECTS SYNTH EVENTS AT THE MOMENT THE CLASS APPEARS
#   - KEEPS FLOW (sip/dip/dport) from a REAL sample of that class
# ==========================================================

def build_smote_generator_for_class(X, Y, target_class, k_neighbors=3, random_state=42):
    """
    Returns a function gen(x_i) -> synthetic_x
    SMOTE formula: x_new = x_i + r*(x_nn - x_i)
    where nn is chosen among k nearest neighbors in same class.
    """
    rng = np.random.default_rng(random_state)
    idx = np.where(Y == target_class)[0]
    if len(idx) < 2:
        return None  # not enough to SMOTE

    Xc = X[idx]
    nn = NearestNeighbors(n_neighbors=min(k_neighbors + 1, len(idx)))  # +1 because itself
    nn.fit(Xc)

    def gen(x_i):
        # find neighbors inside class
        dist, neigh = nn.kneighbors(x_i.reshape(1, -1), return_distance=True)
        neigh = neigh[0]
        # remove itself if present at position 0
        if len(neigh) > 1:
            neigh = neigh[1:]
        # pick one neighbor
        j = rng.integers(0, len(neigh))
        x_j = Xc[neigh[j]]
        r = rng.random()
        return (x_i + r * (x_j - x_i)).astype(np.float32)

    return gen


def oversample_streaming_events(
    X, Y, sip, dip, dport,
    target_classes=(8, 9),
    target_count=1000,
    k_neighbors=3,
    random_state=42
):
    """
    Streaming oversampling:
    - keep original order of X,Y
    - when encountering a sample from rare class c, inject synthetic samples right after it
      until reaching target_count for that class.
    - synthetic sample keeps same sip/dip/dport as the triggering real sample (flow preserved)
    """
    counter = Counter(Y)
    need = {c: max(0, target_count - counter.get(c, 0)) for c in target_classes}
    print("Before streaming oversample:", counter)
    print("Need to add:", need)

    # build generators per class (SMOTE feature space)
    gens = {}
    for c in target_classes:
        gens[c] = build_smote_generator_for_class(X, Y, c, k_neighbors=k_neighbors, random_state=random_state + int(c))

    X_out, Y_out, sip_out, dip_out, dport_out = [], [], [], [], []

    added = {c: 0 for c in target_classes}

    # iterate in original temporal order
    for i in tqdm(range(len(X)), desc="Streaming oversample + sequence preparation base list"):
        x_i = X[i]
        y_i = int(Y[i])

        # keep original event
        X_out.append(x_i)
        Y_out.append(y_i)
        sip_out.append(int(sip[i]))
        dip_out.append(int(dip[i]))
        dport_out.append(int(dport[i]))

        # if this is one of the target classes, maybe inject some synthetic events
        if y_i in need and need[y_i] > 0:
            gen = gens.get(y_i, None)
            if gen is None:
                continue

            # inject ONE synthetic right after this real one (repeat over time as we see them)
            # so we don't create huge blocks that distort chronology too much.
            x_syn = gen(x_i)

            X_out.append(x_syn)
            Y_out.append(y_i)

            # IMPORTANT: keep the SAME flow couple for chronology/flow buffer
            sip_out.append(int(sip[i]))
            dip_out.append(int(dip[i]))
            dport_out.append(int(dport[i]))

            need[y_i] -= 1
            added[y_i] += 1

    print("Added:", added)
    print("After streaming oversample:", Counter(Y_out))

    return (
        np.asarray(X_out, dtype=np.float32),
        np.asarray(Y_out, dtype=np.int64),
        np.asarray(sip_out, dtype=np.int64),
        np.asarray(dip_out, dtype=np.int64),
        np.asarray(dport_out, dtype=np.int64),
    )


# ==========================================================
# SEQUENCE TRANSFO (STREAMING) — KEEP ORDER
#   - uses sip/dip/dport flow key
# ==========================================================

def transfo_streaming(X, sip, dip, dport, d_historique=20):
    flows = []     # list of matrices per flow (82, d_historique)
    hashes = []    # sorted list of (hash, idx_in_flows)
    count_sample = []
    data_input = []

    for i in tqdm(range(len(X)), desc="transfo streaming"):
        raw = X[i]
        key = hash_flow(int(sip[i]), int(dip[i]), int(dport[i]))

        idx = binary_search_hash(hashes, key)
        if idx != -1:
            nb = count_sample[idx]
            if nb < d_historique:
                pattern = np.hstack((raw[:, None], flows[idx][:, :nb]))
                num_repeats = d_historique // pattern.shape[1]
                remaining = d_historique % pattern.shape[1]
                repeated = np.tile(pattern, (1, num_repeats))
                if remaining > 0:
                    repeated = np.hstack((repeated, pattern[:, :remaining]))
                count_sample[idx] += 1
            else:
                repeated = np.hstack((raw[:, None], flows[idx]))[:, :d_historique]

            flows[idx] = repeated
            data_input.append(repeated)

        else:
            new_mat = np.tile(raw, (d_historique, 1)).T  # (82, 20)
            flows.append(new_mat)
            data_input.append(new_mat)

            hashes.append((key, len(flows) - 1))
            count_sample.append(1)
            hashes.sort(key=lambda x: x[0])

    return np.asarray(data_input, dtype=np.float32)


def split_npy_save(array, number_of_files, folder):
    os.makedirs(folder, exist_ok=True)
    mem = 0
    n = len(array)
    for i in range(number_of_files):
        chunk = array[mem:int((i + 1) * n / number_of_files)]
        np.save(f'./{folder}/X_input_{i}.npy', chunk.astype(np.float32))
        mem = int((i + 1) * n / number_of_files)


# ==========================================================
# PIPELINE
# ==========================================================

if __name__ == "__main__":

    print("--------------------Importation données--------------------")
    df = importation_csv()

    print("--------------------Séparation des données--------------------")
    X_data, Y_data, sip_data, dip_data, sport_data, dport_data = creation_X_Y_ip(df)

    print("--------------------Train/test split 70/30--------------------")
    X_train, X_test, Y_train, Y_test, sip_train, sip_test, dip_train, dip_test, sport_train, sport_test, dport_train, dport_test = \
        choix_donnees_entrainement_70_30(X_data, Y_data, sip_data, dip_data, sport_data, dport_data)

    # ======================================================
    # STREAMING OVERSAMPLE TRAIN ONLY (classes 8 & 9 -> 1000)
    # ======================================================

    print("\n--------------------Streaming oversample train (8/9)--------------------")
    X_os, Y_os, sip_os, dip_os, dport_os = oversample_streaming_events(
        X_train, Y_train,
        sip_train, dip_train, dport_train,
        target_classes=(8, 9),
        target_count=100,
        k_neighbors=3,
        random_state=42
    )

    # ======================================================
    # SEQUENCE BUILDING (TRAIN)
    # ======================================================

    d_historique = 20
    print("\n--------------------Création des séquences TRAIN--------------------")
    X_input_train = transfo_streaming(X_os, sip_os, dip_os, dport_os, d_historique=d_historique)
    X_input_test = transfo_streaming(X_test, sip_test, dip_test, dport_test, d_historique=d_historique)

    split_npy_save(X_input_train, 20, 'X_input_split_train_sequence_port_smote')
    split_npy_save(X_input_test, 20, 'X_input_split_test_sequence_port_smote')

    np.save('./X_input_split_train_sequence_port_smote/Y.npy', Y_os.astype(np.int32))
    np.save('./X_input_split_test_sequence_port_smote/Y.npy', Y_test.astype(np.int32))